In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib
import spacy

# 1. Préparation du NLP (Stop Words Français)
try:
    nlp = spacy.load("fr_core_news_lg")
except:

    print("Erreur: Modèle SpaCy fr_core_news_lg non trouvé.")
    
stop_words_french = list(nlp.Defaults.stop_words)

# 2. Chargement des données (Séparateur ;)
# 500 premières lignes
file_path = 'articles_all.csv'
df = pd.read_csv(file_path, sep=';', encoding='utf-8').head(500)

# 3. Nettoyage 
df['title'] = df['title'].fillna('')
df['tags'] = df['tags'].fillna('')
df['description'] = df['description'].fillna('')

# On transforme les vides et les espaces en 0 avant de convertir en int
df['label'] = df['label'].replace(['', ' ', None], 0)
df['label'] = df['label'].fillna(0).astype(int)

# Vérification du nombre d'exemples par classe
print("Distribution des classes dans les 500 lignes :")
print(df['label'].value_counts())
print("-" * 30)

# 4. Préparation de X (Features) et y (Target)
# On fusionne Titre + Tags + Description pour donner un maximum de contexte
X = df['title'] + " " + df['tags'] + " " + df['description']
y = df['label']

# 5. Création du Pipeline
# ngram_range=(1, 2) permet d'analyser les mots seuls et les couples de mots (ex: "saisie drogue")
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2), 
        max_features=10000, 
        stop_words=stop_words_french
    )),
    ('clf', LinearSVC(class_weight='balanced', C=1.0, random_state=42))
])

# 6. Entraînement et Évaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Entraînement en cours...")
pipeline.fit(X_train, y_train)

# Affichage des performances
y_pred = pipeline.predict(X_test)
print("\nRAPPORT DE PERFORMANCE :")
print(classification_report(y_test, y_pred))

# 7. Sauvegarde du modèle final
# On ré-entraîne sur les 500 lignes complètes pour ne perdre aucune info
pipeline.fit(X, y)
model_filename = 'modele_securite_v2.joblib'
joblib.dump(pipeline, model_filename)

print(f"\nSuccès ! Modèle sauvegardé sous : {model_filename}")

# --- PETIT TEST RAPIDE ---
test_text = "Arrestation d'un dealer avec 500 comprimés de ecstasy à l'Aouina"
prediction = pipeline.predict([test_text])[0]
print(f"Test prediction: {test_text} --> Catégorie: {prediction}")

Distribution des classes dans les 500 lignes :
label
0    352
1     60
2     53
3     21
4     14
Name: count, dtype: int64
------------------------------
Entraînement en cours...


C:\Users\Pc\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(



RAPPORT DE PERFORMANCE :
              precision    recall  f1-score   support

           0       0.90      0.94      0.92        66
           1       0.82      0.90      0.86        10
           2       0.67      0.77      0.71        13
           3       1.00      0.50      0.67         6
           4       1.00      0.40      0.57         5

    accuracy                           0.86       100
   macro avg       0.88      0.70      0.75       100
weighted avg       0.87      0.86      0.85       100


Succès ! Modèle sauvegardé sous : modele_securite_v2.joblib
Test prediction: Arrestation d'un dealer avec 500 comprimés de ecstasy à l'Aouina --> Catégorie: 1


In [7]:
import pandas as pd
import joblib
import spacy
import re

# 1. Chargement des outils
model = joblib.load('modele_securite_v2.joblib')
nlp = spacy.load("fr_core_news_lg")

# 2. Lecture du fichier avec le bon encodage pour les accents
df_all = pd.read_csv('articles_all.csv', sep=';', encoding='utf-8')
df_rest = df_all.iloc[500:].copy()

def extraire_date_evoluee(row):
    # On cherche dans le titre, la description et l'URL (souvent la date est dans l'URL)
    texte_source = f"{row['title']} {row['description']} {row.get('url', '')}"
    
    # Pattern pour dates : 25/03/2026 ou 2026/03/25 ou 25-03-2026
    match = re.search(r'(\d{2,4}[/-]\d{1,2}[/-]\d{2,4})', str(texte_source))
    if match:
        return match.group(1)
    
    # Si on ne trouve rien, on peut mettre "Date non mentionnée"
    return "Date inconnue"

results = []

print("Analyse en cours...")
for _, row in df_rest.iterrows():
    texte_complet = f"{row['title']} {row['tags']} {row['description']}"
    cat_predite = model.predict([texte_complet])[0]
    
    if cat_predite > 0:
        doc = nlp(texte_complet)
        # Nettoyage des lieux pour enlever les caractères spéciaux si besoin
        lieux = list(set([ent.text for ent in doc.ents if ent.label_ == "LOC"]))
        
        results.append({
            'titre': str(row['title']),
            'categorie': int(cat_predite),
            'lieux_extraits': ", ".join(lieux),
            'date': extraire_date_evoluee(row),
            'url': row.get('url', '')
        })

# 3. Sauvegarde propre
df_final = pd.DataFrame(results)

# On utilise utf-8-sig pour que Excel reconnaisse les accents immédiatement
df_final.to_csv('analyse_propre.csv', index=False, sep=';', encoding='utf-8-sig')

print(f"Terminé ! Fichier 'analyse_propre.csv' généré sans index et avec accents corrigés.")

Analyse en cours...


C:\Users\Pc\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['neuf', 'qu', 'quelqu'] not in stop_words.
  warnings.warn(


Terminé ! Fichier 'analyse_propre.csv' généré sans index et avec accents corrigés.
